# E7 — DDR

The full 384 px field and two 224 px detail views test whether retaining more local information improves validation macro-F1. No result in this checkout tests that hypothesis.

[Notebook index](../README.md) · [Experiment audit](../../docs/NOTEBOOK_EXPERIMENT_AUDIT.md)

## Protocol and reproducibility

[Runner](../../novelty_integrated/e7/runner.py) · [Protocol and environment](../../novelty_integrated/e7/README.md) · [geometry.py](../../novelty_integrated/e7/geometry.py) · [model.py](../../novelty_integrated/e7/model.py) · [input_pipeline.py](../../novelty_integrated/e7/input_pipeline.py) · [config.py](../../novelty_integrated/e7/config.py)

Frozen seed 42 manifest, SHA group isolation, grade order 0–4. Validation uses deterministic transforms and original support. The notebook creates no split.

## Model, inputs and environment

Official Hugging Face ConvNeXt-Tiny V1, pooled width 768. A categorical head emits five logits and an ordered-threshold ordinal head contributes auxiliary supervision. One shared backbone serves the full 384×384 view and two 224×224 patches. A 1,537-parameter gate fuses the pooled global and mean local embeddings. The external checkpoint has no verified count in this checkout.

EXIF-corrected original RGB is letterboxed to 384 px. Two crops use 55% of the estimated field width and height, are letterboxed to 224 px, and use official normalization with resize disabled. B1 train-only augmentation is keyed by seed, epoch and image ID. It applies horizontal flip (p=0.50), field-of-view guarded rotation (p=0.50, ±7°), brightness (p=0.30, factor 0.90–1.10), and contrast (p=0.30, factor 0.90–1.10), in that order. Decoded train RGB can reuse compatible E6 cache read-only, and validation uses an isolated E7 cache.

The evidence view requires Python, Jupyter and IPython. Optional source workflows rely on the run-specific PyTorch, Transformers, Pillow and NumPy environment, a verified local snapshot, and external datasets/manifests. The CLI and protocol links describe hardware, caching and environment details.

### Read committed evidence

The setup uses the dataset-specific result and performs no training or inference.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

from IPython.display import JSON, display

configured_root = os.environ.get("DR_RESEARCH_ROOT")
start = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "notebooks/experiment_workflow.py").is_file() and (p / "novelty_integrated").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DR_RESEARCH_ROOT to the reviewed repository checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

METHOD, DATASET = 'E7', 'DDR'
RUNS = []
CONFIG = {'manifest': {'counts': {'train': 10018, 'val': 1252}, 'dataset': 'DDR', 'grade_support': {'train': [5014, 504, 3581, 189, 730], 'val': [626, 63, 448, 24, 91]}, 'group_count': {'train': 9937, 'val': 1241}, 'image_order_sha256': 'b3dc145d12e4554f9210f581f04d6cb06b7ecceb5812b793c5ba9793d244503b', 'images_verified': True, 'manifest_sha256': '7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1', 'manifest_sidecar_sha256': '56270be9a922565e7b4f4bc192fafcc6ebb53110904b3942703f273aa20ae08c'}, 'snapshot': {'config_sha256': '441f982e839a9ea96df7c1678422dc4e65cd5632731a6957f6aa03728195423d', 'hub_id': 'facebook/convnext-tiny-224', 'processor_config_sha256': 'fd575b890da5a949493e1d1e7a70bfcb9e4b99fe444004d2dbfa253add254741', 'revision': '6166b7613034066690a621d8bf25ffdf181a34f0', 'weight_file': 'pytorch_model.bin', 'weight_sha256': '1fe15b2eba9e355de14c4e244e6365bf61ffef84da7d3442c0afbb2d8564f477'}, 'micro_batch': 4, 'accumulation': 8, 'precision': 'bf16', 'input_transport': {'num_workers': 2, 'persistent_workers': True, 'prefetch_factor': 2, 'pin_memory': True}, 'e5_checkpoint_sha256': '7dd7488013b7af5ad4565052b47d203c8077d8203dd5520d5b20b0fe592fbe00'}
CHECKPOINT_SHA256 = None
print(f'{METHOD} {DATASET}: {len(RUNS)} published run(s) in this notebook.')

### Optional manifest and checkpoint byte checks

Set `DR_ARTIFACT_ROOT` and optionally `DR_CHECKPOINT` before enabling this read-only gate.

In [ ]:
VERIFY_LOCAL_PROVENANCE = False
if VERIFY_LOCAL_PROVENANCE:
    artifact = os.environ.get("DR_ARTIFACT_ROOT")
    checkpoint = os.environ.get("DR_CHECKPOINT")
    expected = CONFIG.get("manifest", {}).get("manifest_sha256")
    if not artifact or not expected:
        raise ValueError("Set DR_ARTIFACT_ROOT and use a notebook with a verified frozen manifest.")
    manifest = Path(artifact) / "cad_vit_runs" / "manifests" / f"{DATASET}_seed42.csv"
    with manifest.open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != expected:
            raise ValueError("Frozen manifest hash differs.")
    print("Frozen manifest SHA-256 matches.")
    if checkpoint:
        run_id = os.environ.get("DR_RECORDED_RUN_ID")
        receipts = {r.get("id", r.get("experiment")):
                    r.get("source_sha256", {}).get("best.pt") or
                    r.get("checkpoint_sidecars", {}).get("best.pt", {}).get("sha256") for r in RUNS}
        expected_checkpoint = receipts.get(run_id) if run_id else CHECKPOINT_SHA256
        if len(RUNS) > 1 and not run_id:
            raise ValueError("Set DR_RECORDED_RUN_ID to the relevant B4-Q or B4-R evidence ID.")
        if not expected_checkpoint:
            raise ValueError("No repository-verified checkpoint SHA-256 exists for this run.")
        with Path(checkpoint).open("rb") as stream:
            if hashlib.file_digest(stream, "sha256").hexdigest() != expected_checkpoint:
                raise ValueError("Checkpoint hash differs.")
        print("Checkpoint SHA-256 matches without deserializing weights.")

## Evidence status

No finalized E7 history, validation summary, confusion matrix, checkpoint receipt or training curve is published here. Current HPC execution status is unverified. The **planned** 20-epoch seed-42 pilot uses a dataset-matched E5 warm start, full fine-tuning, effective batch 32, AdamW (backbone 1e−5, head 1e−3, decay 0.01), no scheduler, and validation macro-F1 with validation CE and earlier-epoch tie breaks. Training uses ordinary CE with deferred inverse-square-root weighting from epoch 11 and 0.25 ordinal BCE. The warm-start SHA is from the committed E5 evidence. It is distinct from an E7 checkpoint identity. The linked runner also requires prespecifying whether raw or optional EMA weights generate validation predictions.

## Interpretation and limits

The 384-pixel field covers peripheral content outside the local crops. Patch usefulness remains a hypothesis until validation artifacts are verified.

Do not infer a gain from geometry or multi-view fusion before validation artifacts are verified.

## Optional execution of the authentic workflow

Set the external `DR_ARTIFACT_ROOT`, `DR_CONVNEXT_SNAPSHOT` and for writing steps `DR_NOTEBOOK_RUN_DIR`. Use the protocol's reviewed cache roots, size budgets and paths when applicable. `DR_MANIFEST_SOURCE_ROOT` supports moved roots for E5–E7. Authorize the local snapshot explicitly via `DR_TRUST_LEGACY_BIN=1` where required. GPU actions also require `DR_GPU_ALLOCATION_CONFIRMED=1`. For E7, use a fresh output directory below the external `foundation_e7` root. 

The [shared workflow bridge](../experiment_workflow.py) lists the exact environment keys and dispatches to the original CLI. Choose one step and enable its switch. Outputs must use a fresh external location. Report identity and cache validity remain enforced by the runner.

In [ ]:
from notebooks.experiment_workflow import run_step

ACTION = None
RUN_PREFLIGHT = False
RUN_CACHE = False
RUN_INPUT_VERIFICATION = False
RUN_GPU_SMOKE = False
RUN_BENCHMARK = False
RUN_TRAINING = False
RUN_CHECKPOINT_EXPORT = False
RUN_PRUNING = False
RUN_FINAL_TEST = False
if RUN_CHECKPOINT_EXPORT or RUN_PRUNING or RUN_FINAL_TEST:
    raise ValueError("Export, pruning and final-test inference are outside this workflow.")
run_step(ROOT, METHOD, DATASET, ACTION, CONFIG, {
    "preflight": RUN_PREFLIGHT, "estimate-cache": RUN_PREFLIGHT, "prepare-cache": RUN_CACHE,
    "verify-inputs": RUN_INPUT_VERIFICATION, "smoke": RUN_GPU_SMOKE,
    "benchmark": RUN_BENCHMARK, "train": RUN_TRAINING,
})